# Moodify Project Health Analysis - 2026-08-02

## tl;dr

This notebook measures repository capitalisation, task flow, test trust, change concentration, and data-quality limits. It intentionally does not treat file counts or test counts as product value.

## Context & Methods

The decision is whether Moodify should continue feature expansion or pause for stabilisation. Sources are the local Git worktree, task-package documents, and pytest collection.

### Key Assumptions

- A task with a Codex final-acceptance document is treated as accepted, even if an older handoff still says review-ready.
- Capability Accretion 017-021 packages without handoff/progress are treated as planned.
- Test-file ratio is structural evidence, not code coverage.
- Untracked Git status entries may represent whole directories, so they are reported as entries rather than files.

In [1]:
from collections import Counter
from pathlib import Path
import json, re, subprocess

ROOT = Path(r'E:/moodify')
CORE = ROOT / 'moodify-core-package'
TASKS = ROOT / 'docs/tasks/deepseek'

def run(*args, cwd=ROOT, timeout=120):
    return subprocess.run(args, cwd=cwd, text=True, encoding='utf-8', errors='replace', capture_output=True, timeout=timeout)

def git(*args):
    result = run('git', *args)
    if result.returncode:
        raise RuntimeError(result.stderr)
    return result.stdout

print('Repository:', ROOT)
print('HEAD:', git('rev-parse', '--short', 'HEAD').strip())

Repository: E:\moodify
HEAD: df3a8a3


## Data

### 1. Profile repository and change surface

In [2]:
status_lines = [line for line in git('status', '--porcelain=v1').splitlines() if line]
tracked_files = [line for line in git('ls-files').splitlines() if line]
modified_entries = sum(1 for line in status_lines if not line.startswith('??'))
untracked_entries = sum(1 for line in status_lines if line.startswith('??'))

numstat = git('diff', '--numstat').splitlines()
insertions = deletions = binary_changes = 0
changed_by_area = Counter()
for line in numstat:
    added, removed, path = line.split('\t', 2)
    if added == '-' or removed == '-':
        binary_changes += 1
    else:
        insertions += int(added); deletions += int(removed)
    changed_by_area[path.split('/', 1)[0]] += 1

repo_profile = {
    'tracked_files': len(tracked_files),
    'modified_tracked_entries': modified_entries,
    'untracked_entries': untracked_entries,
    'changed_tracked_files_numstat': len(numstat),
    'insertions': insertions,
    'deletions': deletions,
    'change_lines': insertions + deletions,
    'binary_changes': binary_changes,
    'dirty_tracked_share_pct': round(100 * modified_entries / len(tracked_files), 2),
}
repo_profile, changed_by_area.most_common(10)

({'tracked_files': 2140,
  'modified_tracked_entries': 55,
  'untracked_entries': 136,
  'changed_tracked_files_numstat': 55,
  'insertions': 2390,
  'deletions': 605,
  'change_lines': 2995,
  'binary_changes': 0,
  'dirty_tracked_share_pct': 2.57},
 [('moodify_runtime', 31),
  ('moodify-core-package', 15),
  ('treatment_records', 2),
  ('.gitignore', 1),
  ('CHANGELOG.md', 1),
  ('PROJECT_ROADMAP.md', 1),
  ('README.md', 1),
  ('docs', 1),
  ('scripts', 1),
  ('workers', 1)])

### 2. Measure code and test structure

In [3]:
python_paths = [ROOT / p for p in tracked_files if p.endswith('.py')]
test_paths = [p for p in python_paths if p.name.startswith('test_') or '/tests/' in p.as_posix()]
source_paths = [p for p in python_paths if p not in test_paths]

def physical_lines(paths):
    total = 0
    readable = 0
    for path in paths:
        if path.is_file():
            total += len(path.read_text(encoding='utf-8', errors='replace').splitlines())
            readable += 1
    return readable, total

source_file_count, source_lines = physical_lines(source_paths)
test_file_count, test_lines = physical_lines(test_paths)
code_profile = {
    'python_source_files': source_file_count,
    'python_test_files': test_file_count,
    'source_physical_lines': source_lines,
    'test_physical_lines': test_lines,
    'test_to_source_line_ratio_pct': round(100 * test_lines / source_lines, 1) if source_lines else None,
}
code_profile

{'python_source_files': 203,
 'python_test_files': 112,
 'source_physical_lines': 53119,
 'test_physical_lines': 18157,
 'test_to_source_line_ratio_pct': 34.2}

### 3. Reconcile task-package states using source precedence

In [4]:
task_rows = []
for task_dir in sorted(p for p in TASKS.iterdir() if p.is_dir()):
    if not (task_dir / '00_TASK_ORCHESTRATION.md').exists():
        continue
    acceptance = sorted(task_dir.glob('CODEX_FINAL_ACCEPTANCE*.md'))
    handoff = task_dir / 'HANDOFF.md'
    handoff_status = ''
    if handoff.exists():
        match = re.search(r'^\*\*Status:\*\*\s*(.+?)\s*$', handoff.read_text(encoding='utf-8', errors='replace'), re.M)
        handoff_status = match.group(1).strip() if match else ''
    if acceptance:
        state = 'accepted'
    elif task_dir.name.startswith('DSK-MFY-CAPABILITY-ACCRETION-') and not handoff.exists():
        state = 'planned'
    elif 'NOT_STARTED' in handoff_status:
        state = 'not_started'
    elif 'READY_FOR' in handoff_status:
        state = 'awaiting_acceptance'
    else:
        state = 'unclassified'
    task_rows.append({'task': task_dir.name, 'state': state, 'handoff_status': handoff_status, 'acceptance_docs': len(acceptance)})

all_task_dirs = sorted(p for p in TASKS.iterdir() if p.is_dir())
orphan_task_dirs = [p.name for p in all_task_dirs if not (p / '00_TASK_ORCHESTRATION.md').exists()]
acceptance_without_orchestration = [p.name for p in all_task_dirs if not (p / '00_TASK_ORCHESTRATION.md').exists() and list(p.glob('CODEX_FINAL_ACCEPTANCE*.md'))]
accepted_but_stale_handoff = [row['task'] for row in task_rows if row['acceptance_docs'] and ('READY_FOR' in row['handoff_status'] or 'REWORK' in row['handoff_status'])]
task_states = Counter(row['state'] for row in task_rows)
started = task_states['accepted'] + task_states['awaiting_acceptance']
task_profile = {
    'task_packages': len(task_rows),
    **dict(task_states),
    'accepted_share_of_started_pct': round(100 * task_states['accepted'] / started, 1) if started else None,
    'all_task_directories': len(all_task_dirs),
    'orphan_task_directories': len(orphan_task_dirs),
    'acceptance_without_orchestration': len(acceptance_without_orchestration),
    'accepted_with_stale_handoff': len(accepted_but_stale_handoff),
    'status_source_conflicts': len(accepted_but_stale_handoff) + len(acceptance_without_orchestration),
}
task_profile

{'task_packages': 22,
 'accepted': 8,
 'planned': 5,
 'awaiting_acceptance': 5,
 'not_started': 3,
 'unclassified': 1,
 'accepted_share_of_started_pct': 61.5,
 'all_task_directories': 24,
 'orphan_task_directories': 2,
 'acceptance_without_orchestration': 1,
 'accepted_with_stale_handoff': 3,
 'status_source_conflicts': 4}

### 4. Test trust and reproducibility

In [5]:
collection = run('py', '-3.11', '-m', 'pytest', '--collect-only', '-q', cwd=CORE, timeout=180)
collection_text = collection.stdout + collection.stderr
tests_collected = int(re.search(r'(\d+) tests collected', collection_text).group(1)) if re.search(r'(\d+) tests collected', collection_text) else 0
collection_errors = int(re.search(r'(\d+) errors? in', collection_text).group(1)) if re.search(r'(\d+) errors? in', collection_text) else 0
test_profile = {
    'tests_collected': tests_collected,
    'collection_errors': collection_errors,
    'collection_exit_code': collection.returncode,
    'collection_trust_gate': 'PASS' if collection.returncode == 0 else 'FAIL',
}
test_profile

{'tests_collected': 400,
 'collection_errors': 19,
 'collection_exit_code': 2,
 'collection_trust_gate': 'FAIL'}

## Results

### 5. Compute decision-facing health indicators

In [6]:
indicators = [
    {'dimension': 'Task capitalisation', 'value': task_profile['accepted_share_of_started_pct'], 'unit': '% accepted of started', 'gate': 'WATCH', 'confidence': 'high'},
    {'dimension': 'Test collection trust', 'value': test_profile['collection_errors'], 'unit': 'collection errors', 'gate': 'FAIL' if test_profile['collection_errors'] else 'PASS', 'confidence': 'high'},
    {'dimension': 'Change traceability', 'value': repo_profile['untracked_entries'], 'unit': 'untracked entries', 'gate': 'FAIL' if repo_profile['untracked_entries'] else 'PASS', 'confidence': 'high'},
    {'dimension': 'Change breadth', 'value': len(changed_by_area), 'unit': 'top-level areas touched', 'gate': 'WATCH' if len(changed_by_area) > 3 else 'PASS', 'confidence': 'high'},
    {'dimension': 'Test structure', 'value': code_profile['test_to_source_line_ratio_pct'], 'unit': '% test/source physical lines', 'gate': 'INFO', 'confidence': 'medium'},
]
indicators

[{'dimension': 'Task capitalisation',
  'value': 61.5,
  'unit': '% accepted of started',
  'gate': 'WATCH',
  'confidence': 'high'},
 {'dimension': 'Test collection trust',
  'value': 19,
  'unit': 'collection errors',
  'gate': 'FAIL',
  'confidence': 'high'},
 {'dimension': 'Change traceability',
  'value': 136,
  'unit': 'untracked entries',
  'gate': 'FAIL',
  'confidence': 'high'},
 {'dimension': 'Change breadth',
  'value': 10,
  'unit': 'top-level areas touched',
  'gate': 'WATCH',
  'confidence': 'high'},
 {'dimension': 'Test structure',
  'value': 34.2,
  'unit': '% test/source physical lines',
  'gate': 'INFO',
  'confidence': 'medium'}]

In [7]:
analysis_snapshot = {
    'as_of': '2026-08-02 Asia/Shanghai',
    'repository': repo_profile,
    'code': code_profile,
    'tasks': task_profile,
    'tests': test_profile,
    'change_areas': [{'area': area, 'changed_files': count} for area, count in changed_by_area.most_common()],
    'indicators': indicators,
}
print(json.dumps(analysis_snapshot, ensure_ascii=False, indent=2))

{
  "as_of": "2026-08-02 Asia/Shanghai",
  "repository": {
    "tracked_files": 2140,
    "modified_tracked_entries": 55,
    "untracked_entries": 136,
    "changed_tracked_files_numstat": 55,
    "insertions": 2390,
    "deletions": 605,
    "change_lines": 2995,
    "binary_changes": 0,
    "dirty_tracked_share_pct": 2.57
  },
  "code": {
    "python_source_files": 203,
    "python_test_files": 112,
    "source_physical_lines": 53119,
    "test_physical_lines": 18157,
    "test_to_source_line_ratio_pct": 34.2
  },
  "tasks": {
    "task_packages": 22,
    "accepted": 8,
    "planned": 5,
    "awaiting_acceptance": 5,
    "not_started": 3,
    "unclassified": 1,
    "accepted_share_of_started_pct": 61.5,
    "all_task_directories": 24,
    "orphan_task_directories": 2,
    "acceptance_without_orchestration": 1,
    "accepted_with_stale_handoff": 3,
    "status_source_conflicts": 4
  },
  "tests": {
    "tests_collected": 400,
    "collection_errors": 19,
    "collection_exit_code": 2,

## Takeaways

- Moodify has substantial implementation and test material, but its current trust gate is red because the full test suite cannot be collected.
- The task portfolio is ahead of governance capacity: accepted work is a majority of started work, but review-ready and planned inventory is accumulating behind an unstable baseline.
- Repository counts are reliable for engineering-capital decisions. They are not sufficient to claim user value, audio-quality improvement, revenue potential, or financial ROI.
- The next measurement improvement should capture task start/end time, owner review time, agent execution time, rework time, acceptance outcome, and user-facing evidence.